# Mock DMI Tutorial

This notebook runs the MicrobioLink domain-motif interaction step on
a tiny mocked dataset.

The example is intentionally minimal and portable:

- the human FASTA is mocked,
- the bacterial Pfam table is mocked,
- the motif rules come from the packaged ELM resources that ship with
  `microbiolink_api`.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists() and (path / 'microbiolink').exists():
            return path
    raise RuntimeError('Could not find the MicrobioLink repository root.')


repo_root = find_repo_root()
workdir = repo_root / 'tutorials' / 'mock_data' / 'runs' / '01_mock_dmi'

if workdir.exists():
    shutil.rmtree(workdir)

workdir.mkdir(parents = True)
repo_root, workdir

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(repo_root)],
    check = True,
    cwd = repo_root,
)

In [ ]:
import pandas as pd

from microbiolink_api import interactions_to_dataframe
from microbiolink_api import predict_domain_motif_interactions

## Step 1: Create mock inputs

The packaged default DMI resources contain real ELM motif-to-domain
rules. This mock host FASTA starts both proteins with `MCR`, which is
enough to trigger one of the packaged motif regexes. The bacterial
side contains a matching `PF02207` Pfam domain.

In [ ]:
            human_fasta_file = workdir / 'mock_human_proteins.fasta'
            bacterial_domain_file = workdir / 'mock_bacterial_domains.tsv'

            human_fasta_file.write_text(
                '>sp|P12345|MOCK_POSITIVE
'
                'MCRAAAAAAAAAAAAAAAAAAAAAAAAAAA
'
                '>sp|Q99999|MOCK_SECOND
'
                'MCRGGGGGGGGGGGGGGGGGGGGGGGGGGG
',
                encoding = 'utf-8',
            )

            bacterial_domain_file.write_text(
                'Entry	Pfam
'
                'BACPROT1	PF02207
',
                encoding = 'utf-8',
            )

            print(human_fasta_file)
            print(bacterial_domain_file)

## Step 2: Predict domain-motif interactions

In [ ]:
interactions = predict_domain_motif_interactions(
    fasta_file = human_fasta_file,
    bacterial_domain_file = bacterial_domain_file,
)

interaction_frame = interactions_to_dataframe(interactions)
interaction_frame

In [ ]:
dmi_table_file = workdir / 'mock_dmi_output.tsv'
dmi_legacy_file = workdir / 'mock_dmi_legacy.csv'

interaction_frame.to_csv(dmi_table_file, sep = '	', index = False)
interaction_frame.to_csv(dmi_legacy_file, sep = ';', index = False, header = False)

dmi_table_file, dmi_legacy_file

## Result

This gives you a small DMI table that can be reused for the AIUPred
and Monte Carlo tutorials.